In [1]:
!pip install -q transformers sentence-transformers torch

In [2]:
from transformers import AutoTokenizer
#Goal here  you take off the "black box" cover to inspect the two core inputs to any LLM: Tokenizers and Embeddings.
#See how text is converted into subword tokens and token IDs.
# Load the tokenizer matching the GPT-2 model
tokenizer = AutoTokenizer.from_pretrained("gpt2")

text = "Large Language Models process tokenization effortlessly!"

# Step A: Split text into token strings
tokens = tokenizer.tokenize(text)

# Step B: Map tokens to vocabulary IDs
input_ids = tokenizer.convert_tokens_to_ids(tokens)

# Step C: Re-decode token IDs back into string text
decoded_text = tokenizer.decode(input_ids)

print("--- TOKENIZER INSPECTION ---")
print("1. Raw Tokens:  ", tokens)
print("2. Token IDs:   ", input_ids)
print("3. Decoded Text:", decoded_text)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

--- TOKENIZER INSPECTION ---
1. Raw Tokens:   ['Large', 'ĠLanguage', 'ĠModels', 'Ġprocess', 'Ġtoken', 'ization', 'Ġeffortlessly', '!']
2. Token IDs:    [21968, 15417, 32329, 1429, 11241, 1634, 42241, 0]
3. Decoded Text: Large Language Models process tokenization effortlessly!


In [4]:
#Compare how different tokenizers (e.g., bert-base-uncased vs gpt2) break down the word "tokenization":
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
gpt_tokenizer = AutoTokenizer.from_pretrained("gpt2")

sample_word = "unpretentiousness"
print("BERT Tokens:", bert_tokenizer.tokenize(sample_word))
print("GPT2 Tokens:", gpt_tokenizer.tokenize(sample_word))

BERT Tokens: ['un', '##pre', '##ten', '##tious', '##ness']
GPT2 Tokens: ['un', 'pret', 'entious', 'ness']


In [5]:
#Extracting Static & Contextual Embeddings
#Inspect how token IDs map to dense vector representations.
import torch
from transformers import AutoModel

# Load the underlying neural network backbone
model = AutoModel.from_pretrained("gpt2")

# Convert input text into PyTorch Tensors
inputs = tokenizer(text, return_tensors="pt")

# Pass inputs through model
with torch.no_grad():
    outputs = model(**inputs)

# Extract Token Embeddings
# 1. Static Embeddings from the Embedding Layer (wte = Word Token Embeddings) [cite: 57]
static_embeddings = model.wte(inputs["input_ids"])

# 2. Contextualized Output Embeddings (Hidden states after passing through all Transformer layers) [cite: 58]
contextual_embeddings = outputs.last_hidden_state

print("--- EMBEDDINGS SHAPES ---")
print("Input IDs Tensor Shape:", inputs["input_ids"].shape) # [batch_size, sequence_length]
print("Static Embedding Shape:", static_embeddings.shape)     # [batch_size, sequence_length, hidden_dim]
print("Contextual Output Shape:", contextual_embeddings.shape) # [batch_size, sequence_length, hidden_dim]

print("\nFirst Token's Vector (first 5 values):", contextual_embeddings[0, 0, :5].numpy())

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

--- EMBEDDINGS SHAPES ---
Input IDs Tensor Shape: torch.Size([1, 8])
Static Embedding Shape: torch.Size([1, 8, 768])
Contextual Output Shape: torch.Size([1, 8, 768])

First Token's Vector (first 5 values): [-0.07377776 -0.06918724 -0.3302879  -0.21906729 -0.11195287]


In [6]:
#Document & Sentence Embeddings (Semantic Search)
#Generate full-sentence contextual embeddings using sentence-transformers for tasks like semantic search.
from sentence_transformers import SentenceTransformer, util

# Load a dedicated sentence embedding representation model [cite: 61]
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# Define sample documents and a query
documents = [
    "Solar power uses energy from the sun to create electricity.",
    "Python is a high-level programming language used in machine learning.",
    "Photovoltaic cells collect sunlight and convert it to power."
]

query = "How do we generate renewable solar energy?"

# Compute embeddings
doc_embeddings = embed_model.encode(documents, convert_to_tensor=True)
query_embedding = embed_model.encode(query, convert_to_tensor=True)

# Compute Cosine Similarities between query and documents [cite: 266]
similarities = util.cos_sim(query_embedding, doc_embeddings)

print("--- SEMANTIC SEARCH RESULTS ---")
print("Query:", query, "\n")
for i, doc in enumerate(documents):
    score = similarities[0][i].item()
    print(f"Doc {i+1} [Similarity: {score:.4f}]: {doc}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

--- SEMANTIC SEARCH RESULTS ---
Query: How do we generate renewable solar energy? 

Doc 1 [Similarity: 0.7288]: Solar power uses energy from the sun to create electricity.
Doc 2 [Similarity: 0.0555]: Python is a high-level programming language used in machine learning.
Doc 3 [Similarity: 0.6787]: Photovoltaic cells collect sunlight and convert it to power.
